# Love's Customer Campaign & Segmentation Analytics

Portfolio reconstruction of the original case-competition analysis using the five source CSVs and the submitted project write-up as validation references.

This notebook focuses on reproducible customer segmentation, campaign KPIs, product sales, channel performance, geographic behavior, and the age-vs.-fuel question. Raw case-competition data is intentionally excluded from the public repository.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
DATA_DIRS = [Path("../data"), Path("data"), Path(".")]

def find_file(*names):
    for folder in DATA_DIRS:
        for name in names:
            path = folder / name
            if path.exists():
                return path
    raise FileNotFoundError(f"Missing source file. Tried: {names}")

lookup = pd.read_csv(find_file("DRIVER_LOOKUP.csv", "DRIVER_LOOKUP-Afnan_Rafid.csv"))
aggs = pd.read_csv(find_file("DRIVER_3_YEAR_AGGS.csv", "DRIVER_3_YEAR_AGGS-Afnan_Rafid.csv"))
campaigns = pd.read_csv(find_file("DRIVER_MONTHLY_CAMPAIGNS.csv", "DRIVER_MONTHLY_CAMPAIGNS (1).csv"))
sales = pd.read_csv(find_file("DRIVER_MONTHLY_PRODUCT_SALES.csv"))
stores = pd.read_csv(find_file("STORE_LOOKUP.csv"))


## 1. Data validation

The project joins the driver tables on `DRIVER_ID` and connects `MOSTVISITEDSTORE` to `STORE_CODE`.


In [ ]:
pd.DataFrame({
    "rows": [len(lookup), len(aggs), len(campaigns), len(sales), len(stores)],
    "unique_drivers": [lookup.DRIVER_ID.nunique(), aggs.DRIVER_ID.nunique(), campaigns.DRIVER_ID.nunique(), sales.DRIVER_ID.nunique(), np.nan]
}, index=["DRIVER_LOOKUP", "DRIVER_3_YEAR_AGGS", "DRIVER_MONTHLY_CAMPAIGNS", "DRIVER_MONTHLY_PRODUCT_SALES", "STORE_LOOKUP"])

assert set(lookup.DRIVER_ID) == set(aggs.DRIVER_ID)
assert not (set(campaigns.DRIVER_ID) - set(lookup.DRIVER_ID))
assert not (set(sales.DRIVER_ID) - set(lookup.DRIVER_ID))
assert lookup.MOSTVISITEDSTORE.isin(stores.STORE_CODE).all()


## 2. Incremental campaign impact

Campaign-month behavior is compared with the prior month for fuel gallons, visits, and merchandise sales.


In [ ]:
campaigns["INC_FUEL"] = campaigns.CAMPAIGNMONTHFUELGALLONS - campaigns.PRIORMONTHFUELGALLONS
campaigns["INC_VISITS"] = campaigns.CAMPAIGNMONTHVISITS - campaigns.PRIORMONTHVISITS
campaigns["INC_MERCH"] = campaigns.CAMPAIGNMONTHMERCHSALES - campaigns.PRIORMONTHMERCHSALES

campaign_impact = campaigns.groupby("AGGNAME").agg(
    incremental_fuel=("INC_FUEL", "sum"),
    incremental_visits=("INC_VISITS", "sum"),
    incremental_merch=("INC_MERCH", "sum"),
    campaign_cost=("CAMPAIGN_COST", "sum")
)
campaign_impact["total_incremental_impact"] = campaign_impact[["incremental_fuel", "incremental_visits", "incremental_merch"]].sum(axis=1)
campaign_impact["impact_per_dollar"] = campaign_impact.total_incremental_impact / campaign_impact.campaign_cost
campaign_impact.round(2)


## 3. Segment join and achieve rates

`Join rate = joined campaigns / campaigns with a valid JOINED status`  
`Achieve rate = achieved campaigns / joined campaigns`

This reproduces the project result that **MediumDiscount** is the strongest engagement segment.


In [ ]:
seg = campaigns.merge(lookup[["DRIVER_ID", "SEGMENTGROUP"]], on="DRIVER_ID", how="left")
segment_kpis = seg.groupby("SEGMENTGROUP").agg(
    valid_campaigns=("JOINED", "count"), joined=("JOINED", "sum"), achieved=("ACHIEVE", "sum"),
    avg_campaign_cost=("CAMPAIGN_COST", "mean"), avg_fuel_gallons=("CAMPAIGNMONTHFUELGALLONS", "mean"),
    avg_merch_sales=("CAMPAIGNMONTHMERCHSALES", "mean")
)
segment_kpis["join_rate"] = segment_kpis.joined / segment_kpis.valid_campaigns
segment_kpis["achieve_rate"] = segment_kpis.achieved / segment_kpis.joined
display(segment_kpis[["joined", "achieved", "join_rate", "achieve_rate", "avg_fuel_gallons", "avg_merch_sales"]].round(4))

segment_kpis[["join_rate", "achieve_rate"]].mul(100).plot(kind="bar", figsize=(9,4))
plt.ylabel("Rate (%)"); plt.xlabel("Segment"); plt.title("Join and Achieve Rates by Segment"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 4. Product sales by segment

Monthly product-sales records are linked to each driver's segment to compare contribution and trends.


In [ ]:
sales["TRANSACTION_MONTH"] = pd.to_datetime(sales.TRANSACTION_MONTH)
sales_seg = sales.merge(lookup[["DRIVER_ID", "SEGMENTGROUP"]], on="DRIVER_ID", how="left")

total_sales = sales_seg.groupby("SEGMENTGROUP").SALESTOTAL.sum().sort_values(ascending=False)
product_mix = sales_seg.groupby(["SEGMENTGROUP", "PRODUCTTYPE"]).SALESTOTAL.sum().unstack(fill_value=0)
display(total_sales.to_frame("total_sales"))
display(product_mix)

monthly = sales_seg.groupby(["TRANSACTION_MONTH", "SEGMENTGROUP"]).SALESTOTAL.sum().unstack().sort_index()
monthly.plot(figsize=(10,4)); plt.title("Monthly Product Sales by Segment"); plt.ylabel("Sales ($)"); plt.tight_layout(); plt.show()


## 5. Merchandise campaign check

The submitted report states that the merchandise campaign produced roughly **+15% merchandise sales, +4% fuel sales, and +9% other sales**. The original final notebook was not preserved, so the exact filtering sequence for those percentages cannot be uniquely recovered.

The calculation below uses a transparent interpretation: compare each Merch Extra campaign month with the prior month by driver and product type.


In [ ]:
campaigns["CAMPAIGNSTARTDATE"] = pd.to_datetime(campaigns.CAMPAIGNSTARTDATE.astype(str).str.replace(" Z", "", regex=False))
campaigns["CAMPAIGN_MONTH"] = campaigns.CAMPAIGNSTARTDATE.dt.to_period("M").dt.to_timestamp("M")
campaigns["PRIOR_MONTH"] = campaigns.CAMPAIGN_MONTH - pd.offsets.MonthEnd(1)
monthly_sales = sales.groupby(["DRIVER_ID", "TRANSACTION_MONTH", "PRODUCTTYPE"], as_index=False).SALESTOTAL.sum()
merch = campaigns[campaigns.AGGNAME.eq("Merch Extra")].copy()

def product_change(rows):
    cur = rows[["DRIVER_ID", "CAMPAIGN_MONTH"]].merge(monthly_sales, left_on=["DRIVER_ID", "CAMPAIGN_MONTH"], right_on=["DRIVER_ID", "TRANSACTION_MONTH"], how="left").groupby("PRODUCTTYPE").SALESTOTAL.sum()
    prev = rows[["DRIVER_ID", "PRIOR_MONTH"]].merge(monthly_sales, left_on=["DRIVER_ID", "PRIOR_MONTH"], right_on=["DRIVER_ID", "TRANSACTION_MONTH"], how="left").groupby("PRODUCTTYPE").SALESTOTAL.sum()
    return pd.concat([prev.rename("prior"), cur.rename("campaign")], axis=1).assign(percent_change=lambda d: (d.campaign/d.prior-1)*100)

product_change(merch).round(2)


## 6. Send-channel effectiveness

Channel success is measured as achievements divided by joins. Receipt has the highest observed rate, but it has a much smaller sample and should be interpreted cautiously.


In [ ]:
valid = seg[seg.JOINED.notna()].copy()
channel = valid.groupby("SEND_CHANNEL").agg(joined=("JOINED", "sum"), achieved=("ACHIEVE", "sum"))
channel["success_rate"] = channel.achieved / channel.joined
display(channel.assign(success_rate=channel.success_rate*100).round(2))


## 7. Historical campaign performance

The historical summary compares rewards, goals achieved, total campaign cost, and participant records.


In [ ]:
historical = campaigns.groupby("AGGNAME").agg(
    total_rewards=("REWARD", "sum"), goals_achieved=("ACHIEVE", "sum"),
    total_cost=("CAMPAIGN_COST", "sum"), participant_records=("DRIVER_ID", "size")
)
historical.round(2)


## 8. Geographic behavior

Division is taken from the driver's most-visited store. The original state table aligns with the driver's `STATE` field rather than the store's state.


In [ ]:
geo = (aggs.merge(lookup[["DRIVER_ID", "SEGMENTGROUP", "MOSTVISITEDSTORE", "STATE"]], on="DRIVER_ID")
       .merge(stores[["STORE_CODE", "DIVISION", "STATE"]].rename(columns={"STATE":"STORE_STATE"}), left_on="MOSTVISITEDSTORE", right_on="STORE_CODE", how="left"))

def top3(df, field):
    out = df.groupby(["SEGMENTGROUP", field]).size().rename("count").reset_index()
    out["pct"] = out["count"] / out.groupby("SEGMENTGROUP")["count"].transform("sum") * 100
    return out.sort_values(["SEGMENTGROUP", "count"], ascending=[True, False]).groupby("SEGMENTGROUP").head(3)

display(top3(geo, "DIVISION").round(2))
display(top3(geo, "STATE").round(2))


## 9. Driver age vs. gallons per visit

The original project used age groups and reported Pearson `r = -0.05`. A driver-level check also produces an essentially zero relationship, supporting the same business conclusion.


In [ ]:
PROJECT_REFERENCE_YEAR = 2024
age = lookup.merge(aggs, on="DRIVER_ID")
age["BIRTHDATE"] = pd.to_datetime(age.BIRTHDATE, errors="coerce")
age["AGE"] = PROJECT_REFERENCE_YEAR - age.BIRTHDATE.dt.year
age["AVG_GALLONS_PER_VISIT"] = age.TOTALFUELGALLONS / age.TOTALFUELTRANSACTIONS.replace(0, np.nan)
correlation = age[["AGE", "AVG_GALLONS_PER_VISIT"]].corr().iloc[0,1]
print("Driver-level Pearson correlation:", round(correlation, 4))


## 10. Business recommendations

- Prioritize **MediumDiscount** for campaign engagement and high-value targeting.
- Use **campaign-specific objectives**: DEF Extra is strong on incremental fuel/visits, while Merch Extra is more merchandise-focused.
- Use **Gallons Extra** as the historical benchmark for engagement and goals achieved.
- Tailor communication channels by segment while considering sample size.
- Prefer behavioral variables over age for targeting.
- Use A/B testing, tiered rewards, and targeted offers to improve campaign efficiency.
- Treat division-level location patterns as more informative than the weaker state-level differences.

See `docs/reconstruction_notes.md` for transparency on which findings reproduce exactly and which depend on assumptions because the original final code was not preserved.
